In [5]:
import tkinter as tk
from collections import deque
import heapq 
import random

TRANG_THAI_BAN_DAU = [
    [1, 2, 3],
    [4, 5, 6],
    [0, 7, 8]
]

KET_QUA_DICH = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

def lam_phang_ma_tran(ma_tran):
    mảng_phẳng = []
    for hang in ma_tran:
        mảng_phẳng.extend(hang)
    return mảng_phẳng

TRANG_THAI_DICH_PHANG = tuple(lam_phang_ma_tran(KET_QUA_DICH))
TRANG_THAI_DAU_PHANG = lam_phang_ma_tran(TRANG_THAI_BAN_DAU)

CAC_HUONG = ["Trai", "Phai", "Len", "Xuong"]
HUONG_NGUOC = {"Trai": "Phai", "Phai": "Trai", "Len": "Xuong", "Xuong": "Len"}

class GiaoDienBFSPuzzle:
    def __init__(self, cua_so):
        self.cua_so = cua_so
        self.cua_so.title("8-Puzzle - Breadth-First Search (BFS) Simulator")

        self.ban_co = list(TRANG_THAI_DAU_PHANG)
        self.so_buoc = 0
        self.huong_vua_di = "Bắt đầu"
        self.dang_tu_dong = False
        self._after_id = None
        self.vi_tri_dong_cac_buoc = {}

        khung_tren = tk.Frame(cua_so)
        khung_tren.pack(pady=10, fill="x")

        self.nhan_trang_thai = tk.Label(
            khung_tren,
            text="Số bước: 0",
            anchor="w",
            font=("Arial", 11)
        )
        self.nhan_trang_thai.pack(side="left", padx=10, fill="x", expand=True)

        tk.Button(khung_tren, text="Chạy Belief Search BFS ", command=self.giai_thuat_toan, bg="#007acc", fg="white", font=("Arial", 10, "bold")).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Dừng Auto", command=self.dung_tu_dong).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Trộn Bàn Cờ Vừa", command=lambda: self.tron(15)).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Đặt Lại", command=self.dat_lai).pack(side="right", padx=5)

        khung_chinh = tk.Frame(cua_so)
        khung_chinh.pack(padx=10, pady=10, fill="both", expand=True)

        khung_left = tk.Frame(khung_chinh)
        khung_left.pack(side="left", anchor="n")

        khung_ban_co = tk.Frame(khung_left, bg="dodger blue", padx=8, pady=8)
        khung_ban_co.pack(anchor="n")

        self.danh_sach_o = []
        for hang in range(3):
            for cot in range(3):
                o = tk.Label(
                    khung_ban_co,
                    text="",
                    width=4,
                    height=2,
                    font=("Arial", 22, "bold"),
                    bg="white",
                    relief="raised",
                    borderwidth=3
                )
                o.grid(row=hang, column=cot, padx=4, pady=4)
                self.danh_sach_o.append(o)
                
        khung_dich = tk.Frame(khung_chinh, bg="#888", padx=10, pady=10)
        khung_dich.pack(side="left", padx=20, anchor="n")

        tk.Label(khung_dich, text="GOAL STATE", font=("Arial", 12, "bold"), bg="#888", fg="white").pack(pady=5)

        luoi_dich = tk.Frame(khung_dich, bg="#888")
        luoi_dich.pack()

        for i, gia_tri in enumerate(TRANG_THAI_DICH_PHANG):
            hang, cot = divmod(i, 3)
            chu = "" if gia_tri == 0 else str(gia_tri)
            mau_nen = "#bbb" if gia_tri != 0 else "#999"

            o_dich = tk.Label(
                luoi_dich,
                text=chu,
                width=4,
                height=2,
                font=("Arial", 16, "bold"),
                bg=mau_nen,
                relief="ridge",
                borderwidth=2
            )
            o_dich.grid(row=hang, column=cot, padx=3, pady=3)
            
        khung_phai = tk.Frame(khung_chinh)
        khung_phai.pack(side="left", padx=10, fill="both", expand=True)

        tk.Label(khung_phai, text="LỘ TRÌNH ĐƯỜNG ĐI BFS TỐI ƯU (LOG)", font=("Arial", 10, "bold"), fg="#007acc").pack(anchor="w", pady=2)
        
        thanh_cuon = tk.Scrollbar(khung_phai)
        thanh_cuon.pack(side="right", fill="y")

        self.o_chu_log = tk.Text(
            khung_phai,
            width=42,
            height=16,
            font=("Courier", 10, "bold"),
            bg="#fdfdfd",
            yscrollcommand=thanh_cuon.set
        )
        self.o_chu_log.pack(side="left", fill="both", expand=True)
        thanh_cuon.config(command=self.o_chu_log.yview)
        self.o_chu_log.tag_config("trang_thai_cu", foreground="#999999", background="")
        self.o_chu_log.tag_config("highlight_dong", foreground="white", background="#007acc")

        cua_so.bind("<Left>",  lambda e: self.xu_ly_phim("Trai"))
        cua_so.bind("<Right>", lambda e: self.xu_ly_phim("Phai"))
        cua_so.bind("<Up>",    lambda e: self.xu_ly_phim("Len"))
        cua_so.bind("<Down>",  lambda e: self.xu_ly_phim("Xuong"))

        self.ve_giao_dien()

    def nap_truoc_toan_bo_hanh_trinh_vao_log(self, hanh_trinh, bat_dau, tong_nut_duyet):
        self.o_chu_log.delete("1.0", tk.END)
        self.vi_tri_dong_cac_buoc.clear()
        
        buoc_tam = 0
        self._ghi_mot_khoi_ma_tran_vao_text(buoc_tam, "Bắt đầu", 0, tong_nut_duyet, bat_dau)
        
        for huong_di, mang_cau_hinh in hanh_trinh:
            buoc_tam += 1
            self._ghi_mot_khoi_ma_tran_vao_text(buoc_tam, huong_di, buoc_tam, tong_nut_duyet, mang_cau_hinh)
            
        self.o_chu_log.tag_add("trang_thai_cu", "1.0", tk.END)

    def _ghi_mot_khoi_ma_tran_vao_text(self, buoc, huong, do_sau, tong_nut, mang):
        m = [str(x) if x != 0 else " " for x in mang]
        danh_dau_dau = self.o_chu_log.index(tk.END + "-1c")
        
        van_ban_ma_tran = (
            f"┌─ Bước {buoc:03d} ────────────────────────┐\n"
            f"│ Hướng dịch: {huong:<7} | Tầng sâu: {do_sau:<5} │\n"
            f"│ Tổng nút quét trong RAM: {tong_nut:<11} │\n"
            f"│ Chiến lược: Belief Goal Search (BDI)  │\n"
            f"│   [ {m[0]} {m[1]} {m[2]} ]                                │\n"
            f"│   [ {m[3]} {m[4]} {m[5]} ]                                │\n"
            f"│   [ {m[6]} {m[7]} {m[8]} ]                                │\n"
            f"└──────────────────────────────────────┘\n"
            f"                   ▼                    \n"
        )
        self.o_chu_log.insert(tk.END, van_ban_ma_tran)
        danh_dau_cuoi = self.o_chu_log.index(tk.END + "-1c")
        
        self.vi_tri_dong_cac_buoc[buoc] = (danh_dau_dau, danh_dau_cuoi)

    def ve_giao_dien(self):
        for i, gia_tri in enumerate(self.ban_co):
            o = self.danh_sach_o[i]
            if gia_tri == 0:
                o.config(text="", bg="#555", relief="sunken")
            else:
                o.config(text=str(gia_tri), bg="#f2f2f2", relief="raised")

        if tuple(self.ban_co) == TRANG_THAI_DICH_PHANG:
            self.nhan_trang_thai.config(text=f"Đã đạt đích tối ưu bằng BDI Belief Search!", fg="green")
        else:
            self.nhan_trang_thai.config(text=f"Số bước dịch chuyển hiện tại: {self.so_buoc}", fg="black")

        if self.so_buoc in self.vi_tri_dong_cac_buoc:
            self.o_chu_log.tag_remove("highlight_dong", "1.0", tk.END)
            dong_dau, dong_cuoi = self.vi_tri_dong_cac_buoc[self.so_buoc]
            self.o_chu_log.tag_add("highlight_dong", dong_dau, dong_cuoi)
            self.o_chu_log.see(dong_dau)
        else:
            self.o_chu_log.tag_remove("highlight_dong", "1.0", tk.END)

    def dat_lai(self):
        self.dung_tu_dong()
        self.ban_co = list(TRANG_THAI_DAU_PHANG)
        self.so_buoc = 0
        self.huong_vua_di = "Bắt đầu"
        self.o_chu_log.delete("1.0", tk.END)
        self.vi_tri_dong_cac_buoc.clear()
        self.ve_giao_dien()

    def xu_ly_phim(self, huong):
        if self.dang_tu_dong or tuple(self.ban_co) == TRANG_THAI_DICH_PHANG:
            return
        kq_moi = self.di_chuyen_ao_toan_bo(self.ban_co, huong)
        if kq_moi:
            self.ban_co = list(kq_moi)
            self.so_buoc += 1
            self.huong_vua_di = huong
            self.ve_giao_dien()

    def di_chuyen_o_trong(self, mang_ban_co, huong):
        vi_tri_trong = mang_ban_co.index(0)
        hang, cot = divmod(vi_tri_trong, 3)
        if huong == "Trai": hang_moi, cot_moi = hang, cot - 1
        elif huong == "Phai": hang_moi, cot_moi = hang, cot + 1
        elif huong == "Len": hang_moi, cot_moi = hang - 1, cot
        elif huong == "Xuong": hang_moi, cot_moi = hang + 1, cot
        else: return False
        if not (0 <= hang_moi < 3 and 0 <= cot_moi < 3): return False
        vi_tri_moi = hang_moi * 3 + cot_moi
        mang_ban_co[vi_tri_trong], mang_ban_co[vi_tri_moi] = mang_ban_co[vi_tri_moi], mang_ban_co[vi_tri_trong]
        return True

    def di_chuyen_ao_toan_bo(self, mang_ban_co, huong):
        vi_tri_trong = mang_ban_co.index(0)
        hang, cot = divmod(vi_tri_trong, 3)
        if huong == "Trai": hang_moi, cot_moi = hang, cot - 1
        elif huong == "Phai": hang_moi, cot_moi = hang, cot + 1
        elif huong == "Len": hang_moi, cot_moi = hang - 1, cot
        elif huong == "Xuong": hang_moi, cot_moi = hang + 1, cot
        else: return None
        if not (0 <= hang_moi < 3 and 0 <= cot_moi < 3): return None
        vi_tri_moi = hang_moi * 3 + cot_moi
        temp = list(mang_ban_co)
        temp[vi_tri_trong], temp[vi_tri_moi] = temp[vi_tri_moi], temp[vi_tri_trong]
        return tuple(temp)

    def tron(self, so_buoc_tron=15):
        self.dung_tu_dong()
        self.ban_co = list(TRANG_THAI_DICH_PHANG)
        self.so_buoc = 0
        self.huong_vua_di = "Trộn"
        self.o_chu_log.delete("1.0", tk.END)
        self.vi_tri_dong_cac_buoc.clear()
        huong_truoc = None
        cac_huong = ["Trai", "Phai", "Len", "Xuong"]
        
        for _ in range(so_buoc_tron):
            random.shuffle(cac_huong)
            for huong in cac_huong:
                if huong_truoc and huong == HUONG_NGUOC[huong_truoc]:
                    continue
                if self.di_chuyen_o_trong(self.ban_co, huong):
                    huong_truoc = huong
                    break
        self.ve_giao_dien()

    def tinh_heuristic_chum_niem_tin(self, tap_niem_tin):
        """Hàm lượng giá Manhattan tổng hợp giúp định hướng chùm niềm tin đi thẳng về đích"""
        tong_manhattan = 0
        for kich_ban in tap_niem_tin:
            for i, gia_tri in enumerate(kich_ban):
                if gia_tri != 0:
                    vi_tri_dich = TRANG_THAI_DICH_PHANG.index(gia_tri)
                    h_hien_tai, c_hien_tai = divmod(i, 3)
                    h_dich, c_dich = divmod(vi_tri_dich, 3)
                    tong_manhattan += abs(h_hien_tai - h_dich) + abs(c_hien_tai - c_dich)
        return tong_manhattan

    def thuat_toan_bfs(self, belief_state_dau):
        """Đã nâng cấp lõi sang A* Search trên Không gian niềm tin giúp xử lý siêu tốc không bị treo"""
        dem_nut = 0
        f_goc = self.tinh_heuristic_chum_niem_tin(belief_state_dau)
        hang_doi_uu_tien = [(f_goc, dem_nut, belief_state_dau, [])]
        
        da_duyet = {str(belief_state_dau): 0}
        tong_nut_da_quet = 1
        GIOI_HAN_NUT_RAM = 30000 

        while hang_doi_uu_tien:
            _, _, niem_tin_ht, duong_di = heapq.heappop(hang_doi_uu_tien)
            if all(kich_ban == TRANG_THAI_DICH_PHANG for kich_ban in niem_tin_ht):
                return duong_di, tong_nut_da_quet

            if tong_nut_da_quet > GIOI_HAN_NUT_RAM:
                return None, tong_nut_da_quet

            g_score_moi = len(duong_di) + 1

            for huong in CAC_HUONG:
                tap_niem_tin_moi = set()
                hop_le = True
                
                for kich_ban in niem_tin_ht:
                    kq_dich_chuyen = self.di_chuyen_ao_toan_bo(kich_ban, huong)
                    if kq_dich_chuyen is None:
                        hop_le = False  # Vi phạm bộ lọc an toàn -> Hủy nhánh!
                        break
                    tap_niem_tin_moi.add(kq_dich_chuyen)

                if hop_le:
                    niem_tin_moi_chuan_hoa = tuple(sorted(list(tap_niem_tin_moi)))
                    id_chuoi = str(niem_tin_moi_chuan_hoa)
                    
                    if id_chuoi not in da_duyet or g_score_moi < da_duyet[id_chuoi]:
                        da_duyet[id_chuoi] = g_score_moi
                        tong_nut_da_quet += 1
                        
                        h_score = self.tinh_heuristic_chum_niem_tin(niem_tin_moi_chuan_hoa)
                        f_score = g_score_moi + h_score
                        
                        kich_ban_log = list(tap_niem_tin_moi)[0]
                        duong_di_moi = duong_di + [(huong, list(kich_ban_log))]
                        
                        dem_nut += 1
                        heapq.heappush(hang_doi_uu_tien, (f_score, dem_nut, niem_tin_moi_chuan_hoa, duong_di_moi))

        return None, tong_nut_da_quet

    def chay_loi_giai(self, hanh_trinh, delay_ms=300):
        self.dang_tu_dong = True
        
        def step(i):
            if not self.dang_tu_dong: return
            if i >= len(hanh_trinh):
                self.dang_tu_dong = False
                self._after_id = None
                self.ve_giao_dien()
                return

            huong_di, _ = hanh_trinh[i]
            kq_thuc_te = self.di_chuyen_ao_toan_bo(self.ban_co, huong_di)
            if kq_thuc_te:
                self.ban_co = list(kq_thuc_te)
                
            self.so_buoc += 1
            self.huong_vua_di = huong_di
            
            self.nhan_trang_thai.config(text=f"BDI tự động dịch chuyển: Hướng {huong_di}", fg="#007acc")
            self.ve_giao_dien()
            
            self._after_id = self.cua_so.after(delay_ms, lambda: step(i + 1))
            
        step(0)

    def dung_tu_dong(self):
        self.dang_tu_dong = False
        if self._after_id is not None:
            try: self.cua_so.after_cancel(self._after_id)
            except Exception: pass
            self._after_id = None

    def giai_thuat_toan(self):
        if self.dang_tu_dong: return
        
        trang_thai_thuc_te = tuple(self.ban_co)
        idx0 = trang_thai_thuc_te.index(0)
        lst_gia_dinh_2 = list(trang_thai_thuc_te)
        idx_canh = idx0 - 1 if idx0 % 3 != 0 else idx0 + 1
        lst_gia_dinh_2[idx0], lst_gia_dinh_2[idx_canh] = lst_gia_dinh_2[idx_canh], lst_gia_dinh_2[idx0]
        trang_thai_gia_dinh_2 = tuple(lst_gia_dinh_2)
        
        belief_state_dau = tuple(sorted(list(set([trang_thai_thuc_te, trang_thai_gia_dinh_2]))))

        self.nhan_trang_thai.config(text="BDI đang quét thông gian Niềm tin siêu tốc...", fg="purple")
        self.cua_so.update_idletasks()

        hanh_trinh, tong_nut = self.thuat_toan_bfs(belief_state_dau)
        
        self.so_buoc = 0
        self.huong_vua_di = "Bắt đầu"
        
        if hanh_trinh is None:
            self.nhan_trang_thai.config(text=f"BDI không tìm ra đường đi an toàn hợp nhất!", fg="red")
            return
            
        self.nap_truoc_toan_bo_hanh_trinh_vao_log(hanh_trinh, trang_thai_thuc_te, tong_nut)
        self.chay_loi_giai(hanh_trinh)

if __name__ == "__main__":
    cua_so_chinh = tk.Tk()
    GiaoDienBFSPuzzle(cua_so_chinh)
    cua_so_chinh.mainloop()